preprocesamiento de la base de datos seleccionada

## Metodología de Preprocesamiento de Datos

El objetivo de este cuaderno es limpiar, transformar y preparar los datos de la base de datos de caña panelera para su posterior análisis y modelado. Los pasos seguidos son los siguientes:

### 1. Carga y Exploración Inicial de Datos

*   **Conexión a Google Drive**: Se montó Google Drive para acceder al archivo CSV almacenado en él.
*   **Carga del Conjunto de Datos**: Se cargó el archivo `caña_panelera_s.csv` utilizando `pandas.read_csv`. Se especificó el delimitador de punto y coma (`;`) y se indicó que todas las columnas se cargaran inicialmente como tipo `str` para preservar la integridad de los valores numéricos que podrían contener comas como separadores decimales o puntos como separadores de miles.
*   **Inspección Inicial**: Se realizó un `df.info()` para visualizar la estructura del DataFrame, incluyendo el tipo de datos de cada columna y la presencia de valores nulos, así como `df.head()` para observar las primeras filas.

### 2. Selección y Normalización de Columnas

*   **Selección de Características**: Se identificaron y seleccionaron un subconjunto de columnas relevantes para el análisis, creando un nuevo DataFrame (`df_filtered`) para trabajar con ellas.
*   **Normalización de Nombres de Columnas**: Todos los nombres de las columnas seleccionadas se convirtieron a minúsculas para estandarización.
*   **Normalización de Contenido de Texto**: Los valores de todas las columnas de tipo `object` (cadena de texto) se convirtieron a minúsculas para asegurar la consistencia y facilitar el agrupamiento o mapeo de valores únicos.

### 3. Limpieza y Codificación de Variables Categóricas

Para varias columnas categóricas, se siguieron los siguientes pasos generales:

*   **Identificación de Valores Únicos**: Se imprimieron los valores únicos y sus conteos (`.unique()`, `.value_counts()`) para entender la distribución y detectar posibles inconsistencias.
*   **Mapeo Numérico / Codificación**: Se creó un diccionario de mapeo para asignar un valor numérico a cada categoría única.
*   **Aplicación del Mapeo**: Se aplicó este mapeo para crear nuevas columnas codificadas (ej., `_encoded`).
*   **Guardado del Mapeo**: Los diccionarios de mapeo se guardaron como archivos JSON en Google Drive para su uso futuro (por ejemplo, para decodificar predicciones o aplicar el mismo mapeo a nuevos datos).
*   **Eliminación de Columnas Originales**: Las columnas categóricas originales se eliminaron del DataFrame una vez que se creó su versión codificada.

Los detalles de cada columna procesada son:

*   **`Departamento` y `Municipio`**: Se codificaron utilizando un mapeo ordinal basado en el orden alfabético de los nombres únicos.
*   **`Cultivo` y `Estado`**: Se identificó que ambas columnas contenían un único valor constante (`'caña panelera'` y `'establecido'`, respectivamente). Por lo tanto, se eliminaron del DataFrame ya que no aportan variabilidad para el modelado.
*   **`Tiempo de establecimiento`**: Se codificó con un mapeo ordinal que representa rangos de tiempo (0: 'no indica', 1: '0-1 año', 2: '1-5 años', 3: '5-10 años', 4: 'más de 10 años').
*   **`Topografia`**: Se aplicó una etapa de corrección para estandarizar descripciones similares (ej., 'ligeramente ondulado' a 'moderadamente ondulado'). Luego, se aplicó un mapeo ordinal (0: 'no indica', 1: 'plano', 2: 'ondulado', etc.).
*   **`Riego`**: Se simplificó a una codificación binaria: 0 si 'no tiene' o 'no indica' y 1 si tiene algún tipo de riego (gravedad, aspersión, goteo, cañón).
*   **`Drenaje`**: Se codificó con un mapeo ordinal que representa la calidad del drenaje (0: 'no indica', 1: 'mal drenaje', 2: 'regular drenaje', 3: 'buen drenaje').

### 4. Ingeniería de Características para 'Fertilizantes Aplicados'

*   **Limpieza de Texto**: Se creó una función `limpiar_texto` para normalizar la columna 'fertilizantes aplicados'. Esta función convierte el texto a minúsculas, elimina acentos, unifica separadores (comas, guiones, etc.) y elimina espacios extra.
*   **Extracción de Palabras Clave y Creación de Banderas Binarias**: Se definieron diccionarios de palabras clave para identificar la aplicación de diferentes tipos de fertilizantes (ej., fósforo, potasio, calcio, orgánico, KCL y fertilización genérica).
*   **Generación de Columnas Binarias**: Se crearon nuevas columnas binarias (ej., `aplica_p`, `aplica_k`) que indican la presencia (1) o ausencia (0) de cada tipo de fertilización en la descripción original.
*   **Eliminación de Columnas Originales de Fertilizantes**: Las columnas 'fertilizantes aplicados' y su versión limpia temporal se eliminaron.

### 5. Conversión de Tipos de Datos Numéricos y Categóricos Finales

*   **Conversión a `category`**: Las columnas codificadas (`_encoded`) y las nuevas columnas binarias de fertilizantes (`aplica_`) se convirtieron explícitamente al tipo `category` de pandas, optimizando el uso de memoria y preparándolas para modelos que manejan este tipo de datos de manera eficiente.
*   **Verificación Final**: Se realizó un `df.info()` para confirmar los tipos de datos finales y el estado general del DataFrame.

### 6. Guardado del DataFrame Procesado

*   **Exportación a CSV**: El DataFrame final procesado (`df_working_copy4`) se guardó como un nuevo archivo CSV (`df_processed.csv`) en Google Drive, sin incluir el índice del DataFrame, listo para ser utilizado en etapas posteriores de modelado.

### Conectar al drive

In [ ]:
# Conectar al drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# librerías
import pandas as pd
import numpy as np

In [ ]:
# Ruta del archivo CSV
file_path = '/content/drive/MyDrive/Semestre9/agriculturaPredictiva/Final_predictiva/caña_panelera_s.csv'

# Cargar la base de datos, especificando el punto y coma como delimitador
# y que todas las columnas sean de tipo string, para que los puntos sean tenidos en cuenta
df = pd.read_csv(file_path, sep=';', dtype=str, engine='python')

print("Base de datos CSV cargada exitosamente con todas las columnas como string. Primeras 5 filas:")
display(df.head())

Base de datos CSV cargada exitosamente con todas las columnas como string. Primeras 5 filas:


,Secuencial,Fecha de Análisis,Departamento,Municipio,Cultivo,Estado,Tiempo de establecimiento,Topografia,Drenaje,Riego,...,Conductividad electrica,Hierro disponible olsen,Cobre disponible,Manganeso disponible Olsen,Zinc disponible Olsen,Boro disponible,Hierro disponible doble acido,Cobre disponible doble acido,Manganeso disponible doble acido,Zinc disponible doble acido
0,23,21/08/2014,CUNDINAMARCA,VILLETA,Caña Panelera,Establecido,Mas de 10 años,Ondulado,No indica,No Tiene,...,2.123,49.80,3.400,1.9,5.5,0.679,ND,ND,ND,ND
1,76,21/08/2014,CUNDINAMARCA,VILLETA,Caña Panelera,Establecido,Mas de 10 años,Ondulado,No indica,No Tiene,...,0.742,59.1,2.7,6.4,7.4,0.301,ND,ND,ND,ND
2,77,21/08/2014,CUNDINAMARCA,VILLETA,Caña Panelera,Establecido,Mas de 10 años,Ondulado,No indica,No Tiene,...,1.196,100,3.1,3.300,1.799,0.373,ND,ND,ND,ND
3,78,21/08/2014,CUNDINAMARCA,NIMAIMA,Caña Panelera,Establecido,Mas de 10 años,Ondulado,No indica,No Tiene,...,0.310,329,7.800,5.5,2,0.085,ND,ND,ND,ND
4,136,21/08/2014,CUNDINAMARCA,NOCAIMA,Caña Panelera,Establecido,Mas de 10 años,Ondulado,No indica,No Tiene,...,0.247,478.0,6.5,4.699,10.8,0.193,ND,ND,ND,ND


In [ ]:
# Visialización columnas bases de datos
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3258 entries, 0 to 3257
Data columns (total 32 columns):
 #   Column                              Non-Null Count  Dtype 
---  ------                              --------------  ----- 
 0   Secuencial                          3258 non-null   object
 1   Fecha de Análisis                   3258 non-null   object
 2   Departamento                        3258 non-null   object
 3   Municipio                           3258 non-null   object
 4   Cultivo                             3258 non-null   object
 5   Estado                              3258 non-null   object
 6   Tiempo de establecimiento           3258 non-null   object
 7   Topografia                          3258 non-null   object
 8   Drenaje                             3258 non-null   object
 9   Riego                               3258 non-null   object
 10  Fertilizantes aplicados             3258 non-null   object
 11  pH agua:suelo                       3253 non-null   obje

In [ ]:
# Seleccionar columnas por procesar
selected_columns = [
    'Departamento',
    'Municipio',
    'Cultivo',
    'Estado',
    'Tiempo de establecimiento',
    'Topografia',
    'Drenaje',
    'Riego',
    'Fertilizantes aplicados',
    'pH agua:suelo',
    'Materia organica',
    'Fósforo Bray II',
    'capacidad de intercambio cationico',
    'Conductividad electrica',
    'Potasio intercambiable'

]

df_filtered = df[selected_columns].copy()
display(df_filtered.head())

,Departamento,Municipio,Cultivo,Estado,Tiempo de establecimiento,Topografia,Drenaje,Riego,Fertilizantes aplicados,pH agua:suelo,Materia organica,Fósforo Bray II,capacidad de intercambio cationico,Conductividad electrica,Potasio intercambiable
0,CUNDINAMARCA,VILLETA,Caña Panelera,Establecido,Mas de 10 años,Ondulado,No indica,No Tiene,NO FERTILIZAN,7.32,6.258,58.11,21.80,2.123,0.348
1,CUNDINAMARCA,VILLETA,Caña Panelera,Establecido,Mas de 10 años,Ondulado,No indica,No Tiene,SI CADA CORTE,6.44,5.935,43.24,17.63,0.742,0.284
2,CUNDINAMARCA,VILLETA,Caña Panelera,Establecido,Mas de 10 años,Ondulado,No indica,No Tiene,NO FERTILIZAN,5.91,4.688,6.999,18.61,1.196,0.246
3,CUNDINAMARCA,NIMAIMA,Caña Panelera,Establecido,Mas de 10 años,Ondulado,No indica,No Tiene,NO FERTILIZAN,5.63,3.995,21.31,8.906,0.310,0.220
4,CUNDINAMARCA,NOCAIMA,Caña Panelera,Establecido,Mas de 10 años,Ondulado,No indica,No Tiene,NO FERTILIZAN,5.65,4.113,84.32,10.39,0.247,0.435


In [ ]:
# Convertir los nombres de las columnas a minúscula
df_filtered.columns = df_filtered.columns.str.lower()

# Display the new column names to confirm the change
print("Column names after converting to lowercase:")
print(df_filtered.columns)

Column names after converting to lowercase:
Index(['departamento', 'municipio', 'cultivo', 'estado',
       'tiempo de establecimiento', 'topografia', 'drenaje', 'riego',
       'fertilizantes aplicados', 'ph agua:suelo', 'materia organica',
       'fósforo bray ii', 'capacidad de intercambio cationico',
       'conductividad electrica', 'potasio intercambiable'],
      dtype='object')


In [ ]:
# Convert all string columns to lowercase
for col in df_filtered.select_dtypes(include=['object']).columns:
    df_filtered[col] = df_filtered[col].astype(str).str.lower()

display(df_filtered.head())

,departamento,municipio,cultivo,estado,tiempo de establecimiento,topografia,drenaje,riego,fertilizantes aplicados,ph agua:suelo,materia organica,fósforo bray ii,capacidad de intercambio cationico,conductividad electrica,potasio intercambiable
0,cundinamarca,villeta,caña panelera,establecido,mas de 10 años,ondulado,no indica,no tiene,no fertilizan,7.32,6.258,58.11,21.80,2.123,0.348
1,cundinamarca,villeta,caña panelera,establecido,mas de 10 años,ondulado,no indica,no tiene,si cada corte,6.44,5.935,43.24,17.63,0.742,0.284
2,cundinamarca,villeta,caña panelera,establecido,mas de 10 años,ondulado,no indica,no tiene,no fertilizan,5.91,4.688,6.999,18.61,1.196,0.246
3,cundinamarca,nimaima,caña panelera,establecido,mas de 10 años,ondulado,no indica,no tiene,no fertilizan,5.63,3.995,21.31,8.906,0.310,0.220
4,cundinamarca,nocaima,caña panelera,establecido,mas de 10 años,ondulado,no indica,no tiene,no fertilizan,5.65,4.113,84.32,10.39,0.247,0.435


## Limpieza de datos

In [ ]:
for column in df_filtered.columns:
    print(f"Column '{column}': {df_filtered[column].nunique()} unique values")

Column 'departamento': 21 unique values
Column 'municipio': 138 unique values
Column 'cultivo': 1 unique values
Column 'estado': 1 unique values
Column 'tiempo de establecimiento': 5 unique values
Column 'topografia': 12 unique values
Column 'drenaje': 4 unique values
Column 'riego': 6 unique values
Column 'fertilizantes aplicados': 141 unique values
Column 'ph agua:suelo': 479 unique values
Column 'materia organica': 2365 unique values
Column 'fósforo bray ii': 2380 unique values
Column 'capacidad de intercambio cationico': 2657 unique values
Column 'conductividad electrica': 985 unique values
Column 'potasio intercambiable': 757 unique values


### Departamentos

In [ ]:
# Se imprimen valores unicos para saber qué departamentos hay
unique_departamentos = df_filtered['departamento'].unique()
print(unique_departamentos)

['cundinamarca' 'valle del cauca' 'santander' 'boyacá' 'huila' 'antioquia'
 'meta' 'putumayo' 'nariño' 'córdoba' 'bogotá, d.c.' 'amazonas' 'tolima'
 'la guajira' 'risaralda' 'caldas' 'cauca' 'caquetá' 'norte de santander'
 'vichada' 'cesar']


In [ ]:
# Se toman los valores unicos de los departamentos y se convierte a tipo categórical, en números
# Luego se crea un diccionario para saber a que numero cooresponde cada departamento

# Ordenar los departamentos alfabéticamente
sorted_departamentos = sorted(unique_departamentos)

# Crear el diccionario, empezando el mapeo numérico desde 1
department_mapping = {dep: i for i, dep in enumerate(sorted_departamentos, start=1)}

print("Department to numerical category mapping:")
for dep, num in department_mapping.items():
    print(f"{dep}: {num}")

# Apply the mapping to the 'Departamento' column in df_filtered
df_filtered['departamento_encoded'] = df_filtered['departamento'].map(department_mapping)

# Display the head of the DataFrame with the new encoded column
print("\nDataFrame with new 'departamento_encoded' column:")
display(df_filtered[['departamento', 'departamento_encoded']].head())

Department to numerical category mapping:
amazonas: 1
antioquia: 2
bogotá, d.c.: 3
boyacá: 4
caldas: 5
caquetá: 6
cauca: 7
cesar: 8
cundinamarca: 9
córdoba: 10
huila: 11
la guajira: 12
meta: 13
nariño: 14
norte de santander: 15
putumayo: 16
risaralda: 17
santander: 18
tolima: 19
valle del cauca: 20
vichada: 21

DataFrame with new 'departamento_encoded' column:


,departamento,departamento_encoded
0,cundinamarca,9
1,cundinamarca,9
2,cundinamarca,9
3,cundinamarca,9
4,cundinamarca,9


In [ ]:
import json
import os

# Define the directory path and filename for the dictionary
save_directory = '/content/drive/MyDrive/Semestre9/agriculturaPredictiva/Final_predictiva/'
file_name = 'department_mapping.json'
full_save_path = os.path.join(save_directory, file_name)

# Ensure the directory exists (though for MyDrive it usually does)
os.makedirs(save_directory, exist_ok=True)

# Save the department_mapping dictionary to a JSON file
with open(full_save_path, 'w', encoding='utf-8') as f:
    json.dump(department_mapping, f, ensure_ascii=False, indent=4)

print(f"Dictionary successfully saved to: {full_save_path}")

Dictionary successfully saved to: /content/drive/MyDrive/Semestre9/agriculturaPredictiva/Final_predictiva/department_mapping.json


In [ ]:
# Eliminar la columna 'departamento'
df_filtered = df_filtered.drop(columns=['departamento'])
df_filtered.head()

,municipio,cultivo,estado,tiempo de establecimiento,topografia,drenaje,riego,fertilizantes aplicados,ph agua:suelo,materia organica,fósforo bray ii,capacidad de intercambio cationico,conductividad electrica,potasio intercambiable,departamento_encoded
0,villeta,caña panelera,establecido,mas de 10 años,ondulado,no indica,no tiene,no fertilizan,7.32,6.258,58.11,21.80,2.123,0.348,9
1,villeta,caña panelera,establecido,mas de 10 años,ondulado,no indica,no tiene,si cada corte,6.44,5.935,43.24,17.63,0.742,0.284,9
2,villeta,caña panelera,establecido,mas de 10 años,ondulado,no indica,no tiene,no fertilizan,5.91,4.688,6.999,18.61,1.196,0.246,9
3,nimaima,caña panelera,establecido,mas de 10 años,ondulado,no indica,no tiene,no fertilizan,5.63,3.995,21.31,8.906,0.310,0.220,9
4,nocaima,caña panelera,establecido,mas de 10 años,ondulado,no indica,no tiene,no fertilizan,5.65,4.113,84.32,10.39,0.247,0.435,9


### Municipio

In [ ]:
# Se imprimen valores unicos para saber qué municipios hay
unique_municipios = df_filtered['municipio'].unique()
print(unique_municipios)

['villeta' 'nimaima' 'nocaima' 'tuluá' 'güepsa' 'moniquirá'
 'san josé de pare' 'santana' 'chitaraque' 'vélez' 'isnos' 'san roque'
 'acacías' 'quipile' 'mocoa' 'útica' 'quebradanegra' 'vergara' 'la peña'
 'sasaima' 'tocaima' 'guaduas' 'florida' 'sandoná' 'la palma' 'lorica'
 'bogotá,  d.c.' 'suaita' 'leticia' 'togüí' 'barbosa' 'san benito'
 'chipatá' 'caparrapí' 'bituima' 'chaguaní' 'guayabal de síquima'
 'la vega' 'palmira' 'cali' 'dagua' 'ocamonte' 'san agustín' 'san gil'
 'fresno' 'san sebastián de mariquita' 'falan' 'vianí' 'distracción'
 'uribe' 'pueblo rico' 'belén de umbría' 'mistrató' 'valle de san josé'
 'neira' 'guadalupe' 'peñol' 'gámbita' 'páramo' 'rosas' 'la sierra'
 'popayán' 'piendamó' 'el tambo' 'gigante' 'garzón' 'la cumbre'
 'versalles' 'supía' 'bugalagrande' 'riosucio' 'el santuario' 'quinchía'
 'rionegro' 'ancuya' 'florencia' 'belén de los andaquíes' 'curillo'
 'albania' 'san josé del fragua' 'san vicente del caguán' 'puerto rico'
 'valparaíso' 'morelia' 'la montañi

In [ ]:
# Ordenar los municipios alfabéticamente
sorted_municipios = sorted(unique_municipios)
# Crear el diccionario de mapeo, empezando el mapeo numérico desde 1
municipality_mapping = {mun: i for i, mun in enumerate(sorted_municipios, start=1)}

print("Municipality to numerical category mapping:")
for mun, num in municipality_mapping.items():
    print(f"{mun}: {num}")

# Aplicar el mapeo a la columna 'municipio' en df_filtered
df_filtered['municipio_encoded'] = df_filtered['municipio'].map(municipality_mapping)

# Mostrar el head del DataFrame con la nueva columna codificada
print("\nDataFrame with new 'municipio_encoded' column:")
display(df_filtered[['municipio', 'municipio_encoded']].head())

Municipality to numerical category mapping:
acacías: 1
albania: 2
amagá: 3
ancuya: 4
angelópolis: 5
ansermanuevo: 6
anzá: 7
arboledas: 8
balboa: 9
barbosa: 10
belén de los andaquíes: 11
belén de umbría: 12
bituima: 13
bogotá,  d.c.: 14
bugalagrande: 15
cajibío: 16
cali: 17
caparrapí: 18
cartago: 19
chaguaní: 20
charalá: 21
chipatá: 22
chitaraque: 23
cocorná: 24
confines: 25
consacá: 26
convención: 27
curillo: 28
dabeiba: 29
dagua: 30
distracción: 31
ebéjico: 32
el cairo: 33
el paujíl: 34
el peñol: 35
el peñón: 36
el santuario: 37
el tambo: 38
falan: 39
florencia: 40
florida: 41
fresno: 42
frontino: 43
garzón: 44
gigante: 45
ginebra: 46
gonzález: 47
guadalajara de buga: 48
guadalupe: 49
guaduas: 50
guayabal de síquima: 51
guática: 52
gámbita: 53
güepsa: 54
isnos: 55
jamundí: 56
la celia: 57
la cumbre: 58
la florida: 59
la montañita: 60
la palma: 61
la peña: 62
la sierra: 63
la vega: 64
leticia: 65
linares: 66
lorica: 67
manzanares: 68
marsella: 69
mesetas: 70
mistrató: 71
mocoa: 72
mogo

,municipio,municipio_encoded
0,villeta,132
1,villeta,132
2,villeta,132
3,nimaima,77
4,nocaima,78


In [ ]:
import json
import os

# Define the directory path and filename for the dictionary
save_directory = '/content/drive/MyDrive/Semestre9/agriculturaPredictiva/Final_predictiva/'
file_name = 'municipality_mapping.json'
full_save_path = os.path.join(save_directory, file_name)

# Ensure the directory exists (though for MyDrive it usually does)
os.makedirs(save_directory, exist_ok=True)

# Save the municipality_mapping dictionary to a JSON file
with open(full_save_path, 'w', encoding='utf-8') as f:
    json.dump(municipality_mapping, f, ensure_ascii=False, indent=4)

print(f"Dictionary successfully saved to: {full_save_path}")

Dictionary successfully saved to: /content/drive/MyDrive/Semestre9/agriculturaPredictiva/Final_predictiva/municipality_mapping.json


In [ ]:
# Eliminar la columna 'municipio' original
df_filtered = df_filtered.drop(columns=['municipio'])
df_filtered.head()

,cultivo,estado,tiempo de establecimiento,topografia,drenaje,riego,fertilizantes aplicados,ph agua:suelo,materia organica,fósforo bray ii,capacidad de intercambio cationico,conductividad electrica,potasio intercambiable,departamento_encoded,municipio_encoded
0,caña panelera,establecido,mas de 10 años,ondulado,no indica,no tiene,no fertilizan,7.32,6.258,58.11,21.80,2.123,0.348,9,132
1,caña panelera,establecido,mas de 10 años,ondulado,no indica,no tiene,si cada corte,6.44,5.935,43.24,17.63,0.742,0.284,9,132
2,caña panelera,establecido,mas de 10 años,ondulado,no indica,no tiene,no fertilizan,5.91,4.688,6.999,18.61,1.196,0.246,9,132
3,caña panelera,establecido,mas de 10 años,ondulado,no indica,no tiene,no fertilizan,5.63,3.995,21.31,8.906,0.310,0.220,9,77
4,caña panelera,establecido,mas de 10 años,ondulado,no indica,no tiene,no fertilizan,5.65,4.113,84.32,10.39,0.247,0.435,9,78


### Cultivo y estado

In [ ]:
# Se imprimen valores unicos para saber cuantos estados hay
unique_cultivo = df_filtered['cultivo'].unique()
print(unique_cultivo)

['caña panelera']


In [ ]:
# Eliminar la columna 'cultivo ya que es un solo tió de cultivo'
df_filtered = df_filtered.drop(columns=['cultivo'])


In [ ]:
# Se imprimen valores unicos para saber cuantos estados hay
unique_cultivo = df_filtered['estado'].unique()
print(unique_cultivo)

['establecido']


In [ ]:
# Eliminar la columna 'cultivo ya que es un solo tió de estado'
df_filtered = df_filtered.drop(columns=['estado'])

In [ ]:
df_filtered.head()

,tiempo de establecimiento,topografia,drenaje,riego,fertilizantes aplicados,ph agua:suelo,materia organica,fósforo bray ii,capacidad de intercambio cationico,conductividad electrica,potasio intercambiable,departamento_encoded,municipio_encoded
0,mas de 10 años,ondulado,no indica,no tiene,no fertilizan,7.32,6.258,58.11,21.80,2.123,0.348,9,132
1,mas de 10 años,ondulado,no indica,no tiene,si cada corte,6.44,5.935,43.24,17.63,0.742,0.284,9,132
2,mas de 10 años,ondulado,no indica,no tiene,no fertilizan,5.91,4.688,6.999,18.61,1.196,0.246,9,132
3,mas de 10 años,ondulado,no indica,no tiene,no fertilizan,5.63,3.995,21.31,8.906,0.310,0.220,9,77
4,mas de 10 años,ondulado,no indica,no tiene,no fertilizan,5.65,4.113,84.32,10.39,0.247,0.435,9,78


## codificación manejo agronómico


In [ ]:
df_working_copy1 = df_filtered.copy()

### Tiempo de establecimineto

In [ ]:
# Se imprimen valores unicos para saber cuantos estados hay
unique_cultivo = df_working_copy1['tiempo de establecimiento'].unique()
print(unique_cultivo)

['mas de 10 años' 'de 5 a 10 años' 'no indica' 'de 0 a 1 año'
 'de 1 a 5 años']


In [ ]:
# Contar la frecuencia de cada valor único en 'tiempo de establecimiento'
counts_tiempo_establecimiento = df_working_copy1['tiempo de establecimiento'].value_counts()

print("Conteo de valores únicos para 'tiempo de establecimiento':")
print(counts_tiempo_establecimiento)

Conteo de valores únicos para 'tiempo de establecimiento':
tiempo de establecimiento
mas de 10 años    1274
no indica          997
de 1 a 5 años      411
de 0 a 1 año       291
de 5 a 10 años     285
Name: count, dtype: int64


In [ ]:
# Define the numerical categorical mapping for 'tiempo de establecimiento'
tiempo_establecimiento_mapping = {
    'mas de 10 años': 4,
    'no indica': 0,
    'de 1 a 5 años': 2,
    'de 0 a 1 año': 1,
    'de 5 a 10 años': 3
}

# Apply the mapping to create a new encoded column
df_working_copy1['tiempo_establecimiento_encoded'] = df_working_copy1['tiempo de establecimiento'].map(tiempo_establecimiento_mapping)

In [ ]:
df_working_copy1['tiempo_establecimiento_encoded'].value_counts()

,count
tiempo_establecimiento_encoded,
4,1274
0,997
2,411
1,291
3,285


In [ ]:
# Eliminar la columna 'tiempo de establecimiento' original
df_working_copy1 = df_working_copy1.drop(columns=['tiempo de establecimiento'])

# Mostrar las primeras filas del DataFrame para verificar el cambio
print("DataFrame después de eliminar la columna 'tiempo de establecimiento':")
display(df_working_copy1.head())

DataFrame después de eliminar la columna 'tiempo de establecimiento':


,topografia,drenaje,riego,fertilizantes aplicados,ph agua:suelo,materia organica,fósforo bray ii,capacidad de intercambio cationico,conductividad electrica,potasio intercambiable,departamento_encoded,municipio_encoded,tiempo_establecimiento_encoded
0,ondulado,no indica,no tiene,no fertilizan,7.32,6.258,58.11,21.80,2.123,0.348,9,132,4
1,ondulado,no indica,no tiene,si cada corte,6.44,5.935,43.24,17.63,0.742,0.284,9,132,4
2,ondulado,no indica,no tiene,no fertilizan,5.91,4.688,6.999,18.61,1.196,0.246,9,132,4
3,ondulado,no indica,no tiene,no fertilizan,5.63,3.995,21.31,8.906,0.310,0.220,9,77,4
4,ondulado,no indica,no tiene,no fertilizan,5.65,4.113,84.32,10.39,0.247,0.435,9,78,4


In [ ]:
import json
import os

# Define the directory path and filename for the dictionary
save_directory = '/content/drive/MyDrive/Semestre9/agriculturaPredictiva/Final_predictiva/'
file_name = 'tiempo_establecimiento_mapping.json'
full_save_path = os.path.join(save_directory, file_name)

# Ensure the directory exists
os.makedirs(save_directory, exist_ok=True)

# Save the tiempo_establecimiento_mapping dictionary to a JSON file
with open(full_save_path, 'w', encoding='utf-8') as f:
    json.dump(tiempo_establecimiento_mapping, f, ensure_ascii=False, indent=4)

print(f"Dictionary successfully saved to: {full_save_path}")

Dictionary successfully saved to: /content/drive/MyDrive/Semestre9/agriculturaPredictiva/Final_predictiva/tiempo_establecimiento_mapping.json


### Topografía

In [ ]:
df_working_copy2 = df_working_copy1.copy()

In [ ]:
# Se imprimen valores unicos para saber qué valores hay en 'topografia'
unique_topografia = df_working_copy2['topografia'].unique()
print(unique_topografia)

['ondulado' 'plano' 'no indica' 'pendiente' 'ligeramente ondulado'
 'pendiente leve' 'plano y ondulado' 'moderadamente ondulado'
 'pendiente moderada' 'pendiente fuerte' 'ondulado y pendiente'
 'plano y pendiente']


In [ ]:
# Imprimir velores único y cuantos por cada uno
print(df_working_copy1['topografia'].value_counts())

topografia
pendiente                 1193
ondulado                  1174
plano                      546
pendiente moderada         122
moderadamente ondulado     117
no indica                   40
ligeramente ondulado        30
pendiente leve              14
plano y ondulado            11
ondulado y pendiente         6
pendiente fuerte             3
plano y pendiente            2
Name: count, dtype: int64


In [ ]:
# Definir un mapeo para estandarizar los valores de 'topografia'
correction_map_topografia = {
    'ligeramente ondulado': 'moderadamente ondulado',
    'pendiente leve': 'pendiente moderada',
    'plano y ondulado': 'moderadamente ondulado',
    'ondulado y pendiente': 'pendiente moderada',
    'pendiente fuerte': 'pendiente',
    'plano y pendiente': 'pendiente'
}

# Aplicar el mapeo a la columna 'topografia' en df_working_copy2
df_working_copy2['topografia'] = df_working_copy2['topografia'].replace(correction_map_topografia)

# Mostrar el conteo de valores únicos corregidos para verificar
print("Conteo de valores únicos corregidos para 'topografia':")
print(df_working_copy2['topografia'].value_counts())

Conteo de valores únicos corregidos para 'topografia':
topografia
pendiente                 1198
ondulado                  1174
plano                      546
moderadamente ondulado     158
pendiente moderada         142
no indica                   40
Name: count, dtype: int64


In [ ]:
topografia_map = {
    'no indica': 0,
    'plano': 1,
    'ondulado': 2,
    'moderadamente ondulado': 3,
    'pendiente moderada': 4,
    'pendiente': 5
}

In [ ]:
# Aplicar el mapeo para crear la nueva columna codificada
df_working_copy2['topografia_encoded'] = df_working_copy2['topografia'].map(topografia_map)

print("\nConteo de valores únicos de la columna codificada 'topografia_encoded':")
print(df_working_copy2['topografia_encoded'].value_counts(dropna=False))


Conteo de valores únicos de la columna codificada 'topografia_encoded':
topografia_encoded
5    1198
2    1174
1     546
3     158
4     142
0      40
Name: count, dtype: int64


In [ ]:
import json
import os

# Define la ruta del directorio y el nombre del archivo para el diccionario
save_directory = '/content/drive/MyDrive/Semestre9/agriculturaPredictiva/Final_predictiva/'
file_name = 'topografia_mapping.json'
full_save_path = os.path.join(save_directory, file_name)

# Asegúrate de que el directorio exista
os.makedirs(save_directory, exist_ok=True)

# Guarda el diccionario topografia_map en un archivo JSON
with open(full_save_path, 'w', encoding='utf-8') as f:
    json.dump(topografia_map, f, ensure_ascii=False, indent=4)

print(f"Diccionario guardado exitosamente en: {full_save_path}")

Diccionario guardado exitosamente en: /content/drive/MyDrive/Semestre9/agriculturaPredictiva/Final_predictiva/topografia_mapping.json


In [ ]:
# Eliminar la columna 'topografia' original
df_working_copy2 = df_working_copy2.drop(columns=['topografia'])

# Mostrar las primeras filas del DataFrame para verificar el cambio
print("DataFrame después de eliminar la columna 'topografia':")
display(df_working_copy2.head())

DataFrame después de eliminar la columna 'topografia':


,drenaje,riego,fertilizantes aplicados,ph agua:suelo,materia organica,fósforo bray ii,capacidad de intercambio cationico,conductividad electrica,potasio intercambiable,departamento_encoded,municipio_encoded,tiempo_establecimiento_encoded,topografia_encoded
0,no indica,no tiene,no fertilizan,7.32,6.258,58.11,21.80,2.123,0.348,9,132,4,2
1,no indica,no tiene,si cada corte,6.44,5.935,43.24,17.63,0.742,0.284,9,132,4,2
2,no indica,no tiene,no fertilizan,5.91,4.688,6.999,18.61,1.196,0.246,9,132,4,2
3,no indica,no tiene,no fertilizan,5.63,3.995,21.31,8.906,0.310,0.220,9,77,4,2
4,no indica,no tiene,no fertilizan,5.65,4.113,84.32,10.39,0.247,0.435,9,78,4,2


### Riego

In [ ]:
# Se imprimen valores unicos para saber qué valores hay en 'riego'
unique_riego = df_working_copy2['riego'].unique()
print(unique_riego)

['no tiene' 'gravedad' 'no indica' 'aspersión' 'goteo' 'cañon']


In [ ]:
# Contar la frecuencia de cada valor único en 'riego'
counts_riego = df_working_copy2['riego'].value_counts()

print("Conteo de valores únicos para 'riego':")
print(counts_riego)

Conteo de valores únicos para 'riego':
riego
no tiene     2476
no indica     664
gravedad       93
aspersión      18
goteo           4
cañon           3
Name: count, dtype: int64


Debido a que hay muy pocos datos de los que tienen dos tipos de riego se decide tener dos categorías, no tiene o tiene algún tipo de riego, sienfo 0 o 1 respectivamente

In [ ]:
# Definir el mapeo ordinal para 'riego'
riego_map = {
    'no indica': 0,
    'no tiene': 0,
    'goteo': 1,
    'cañón': 1,
    'aspersión': 1,
    'gravedad': 1
}

# Aplicar el mapeo para crear la nueva columna codificada
df_working_copy2['riego_encoded'] = df_working_copy2['riego'].map(riego_map)


In [ ]:
print("\nConteo de valores únicos de la columna codificada 'riego_encoded':")
print(df_working_copy2['riego_encoded'].value_counts(dropna=False))


Conteo de valores únicos de la columna codificada 'riego_encoded':
riego_encoded
0.0    3140
1.0     115
NaN       3
Name: count, dtype: int64


In [ ]:
import json
import os

# Define la ruta del directorio y el nombre del archivo para el diccionario
save_directory = '/content/drive/MyDrive/Semestre9/agriculturaPredictiva/Final_predictiva/'
file_name = 'riego_mapping.json'
full_save_path = os.path.join(save_directory, file_name)

# Asegúrate de que el directorio exista
os.makedirs(save_directory, exist_ok=True)

# Guarda el diccionario riego_map en un archivo JSON
with open(full_save_path, 'w', encoding='utf-8') as f:
    json.dump(riego_map, f, ensure_ascii=False, indent=4)

print(f"Diccionario guardado exitosamente en: {full_save_path}")

Diccionario guardado exitosamente en: /content/drive/MyDrive/Semestre9/agriculturaPredictiva/Final_predictiva/riego_mapping.json


In [ ]:
# Eliminar la columna 'riego' original
df_working_copy2 = df_working_copy2.drop(columns=['riego'])

# Mostrar las primeras filas del DataFrame para verificar el cambio
print("DataFrame después de eliminar la columna 'riego':")
display(df_working_copy2.head())

DataFrame después de eliminar la columna 'riego':


,drenaje,fertilizantes aplicados,ph agua:suelo,materia organica,fósforo bray ii,capacidad de intercambio cationico,conductividad electrica,potasio intercambiable,departamento_encoded,municipio_encoded,tiempo_establecimiento_encoded,topografia_encoded,riego_encoded
0,no indica,no fertilizan,7.32,6.258,58.11,21.80,2.123,0.348,9,132,4,2,0.0
1,no indica,si cada corte,6.44,5.935,43.24,17.63,0.742,0.284,9,132,4,2,0.0
2,no indica,no fertilizan,5.91,4.688,6.999,18.61,1.196,0.246,9,132,4,2,0.0
3,no indica,no fertilizan,5.63,3.995,21.31,8.906,0.310,0.220,9,77,4,2,0.0
4,no indica,no fertilizan,5.65,4.113,84.32,10.39,0.247,0.435,9,78,4,2,0.0


### Drenaje

In [ ]:
# Contar la frecuencia de cada valor único en 'drenaje'
counts_drenaje = df_working_copy2['drenaje'].value_counts()

print("Conteo de valores únicos para 'drenaje':")
print(counts_drenaje)

Conteo de valores únicos para 'drenaje':
drenaje
buen drenaje       1393
no indica          1063
regular drenaje     613
mal drenaje         189
Name: count, dtype: int64


Se decide usar un mapeo ordinal para el drenaje.

In [ ]:
# Definir el mapeo ordinal para 'drenaje'
drenaje_map = {
    'no indica': 0,
    'mal drenaje': 1,
    'regular drenaje': 2,
    'buen drenaje': 3
}

# Aplicar el mapeo para crear la nueva columna codificada
df_working_copy2['drenaje_encoded'] = df_working_copy2['drenaje'].map(drenaje_map)


In [ ]:
print("\nConteo de valores únicos de la columna codificada 'drenaje_encoded':")
print(df_working_copy2['drenaje_encoded'].value_counts(dropna=False))


Conteo de valores únicos de la columna codificada 'drenaje_encoded':
drenaje_encoded
3    1393
0    1063
2     613
1     189
Name: count, dtype: int64


In [ ]:
import json
import os

# Define la ruta del directorio y el nombre del archivo para el diccionario
save_directory = '/content/drive/MyDrive/Semestre9/agriculturaPredictiva/Final_predictiva/'
file_name = 'drenaje_mapping.json'
full_save_path = os.path.join(save_directory, file_name)

# Asegúrate de que el directorio exista
os.makedirs(save_directory, exist_ok=True)

# Guarda el diccionario drenaje_map en un archivo JSON
with open(full_save_path, 'w', encoding='utf-8') as f:
    json.dump(drenaje_map, f, ensure_ascii=False, indent=4)

print(f"Diccionario guardado exitosamente en: {full_save_path}")

Diccionario guardado exitosamente en: /content/drive/MyDrive/Semestre9/agriculturaPredictiva/Final_predictiva/drenaje_mapping.json


In [ ]:
# Eliminar la columna 'drenaje' original
df_working_copy2 = df_working_copy2.drop(columns=['drenaje'])

# Mostrar las primeras filas del DataFrame para verificar el cambio
print("DataFrame después de eliminar la columna 'drenaje':")
display(df_working_copy2.head())

DataFrame después de eliminar la columna 'drenaje':


,fertilizantes aplicados,ph agua:suelo,materia organica,fósforo bray ii,capacidad de intercambio cationico,conductividad electrica,potasio intercambiable,departamento_encoded,municipio_encoded,tiempo_establecimiento_encoded,topografia_encoded,riego_encoded,drenaje_encoded
0,no fertilizan,7.32,6.258,58.11,21.80,2.123,0.348,9,132,4,2,0.0,0
1,si cada corte,6.44,5.935,43.24,17.63,0.742,0.284,9,132,4,2,0.0,0
2,no fertilizan,5.91,4.688,6.999,18.61,1.196,0.246,9,132,4,2,0.0,0
3,no fertilizan,5.63,3.995,21.31,8.906,0.310,0.220,9,77,4,2,0.0,0
4,no fertilizan,5.65,4.113,84.32,10.39,0.247,0.435,9,78,4,2,0.0,0


### Fertilizantes Aplicados

In [ ]:
# Se imprimen valores unicos para saber qué valores hay en 'fertilizantes aplicados'
unique_fertilizantes = df_working_copy2['fertilizantes aplicados'].unique()
print(unique_fertilizantes)

['no fertilizan' 'si cada corte' 'mezclas' 'no indica' 'dap - urea'
 'triple 15-15-15' 'triple 15' 'quimico' 'quimico-organico' 'ninguno'
 'abono organico' 'cal' 'gallinaza' 'no' 'si' 'porquinaza' 'urea'
 '15-15-15' 'urea 15-15-15' '15-15-16' '15-15-17' 'no utiliza'
 'úera-gallinaza-k' 'dap-kcl' 'dap-kcl-produmac' 'si (15-15-15)'
 'si (18-18-18)' 'si (15-15-159' 'si (10-20-20)' 'si (10-20-30)'
 'si (15-15-15) urea' 'si (12-24-24)' 'sdi' 'si  (15-15-15)'
 'si  (18-18-18)' 'si  (10-20-30)' 'si  (10-20-20)' 'dsoi  (15-15-15)'
 'si (18-18-19)' 'caña' 'super café' '10,30,10' 'no fertiliza' 'organico'
 'n-k' 'vinaza' '15x15x15' 'abono 10-20-20' 'boñiga,gallinaza' 'boñiga'
 'aboniza,fulviraiz' 'cal agricola' 'pollinaza,gallinaza' 'aboniza'
 'pollinaza' 'aboniza, gallinaza' 'aboniza,gallinaza' 'aboniza,fosforita'
 '10-20-20 kcl' 'vinurea' 'panser' 'organicos'
 'kcl, urea, gallinaza, roca' 'gallinaza, urea, p,k'
 'kcl, urea, gallinaza, roca fosforia' 'urea, p, kcl'
 'urea-dap-kcl-kieserita-cal 

In [ ]:
# Contar la frecuencia de cada valor único en 'fertilizantes aplicados'
counts_fertilizantes = df_working_copy2['fertilizantes aplicados'].value_counts()

print("Conteo de valores únicos para 'fertilizantes aplicados':")
print(counts_fertilizantes)

Conteo de valores únicos para 'fertilizantes aplicados':
fertilizantes aplicados
no                   1348
no indica             504
ninguno               473
si                    232
vinaza                113
                     ... 
15-15-15 20bts/ha       1
15-15-15-10-10-10       1
10-20-30                1
15:15:15 700kg/ha       1
18-18-18                1
Name: count, Length: 141, dtype: int64


In [ ]:
# hacer una copia
df_working_copy3 = df_working_copy2.copy()

In [ ]:
import pandas as pd
import numpy as np
import unicodedata
import re
# Limpieza texto, caracteres especiales
def limpiar_texto(texto):

    if pd.isna(texto):
        return ''

    texto = str(texto).lower().strip()

    # corregir caracteres raros
    texto = unicodedata.normalize('NFKD', texto)\
                       .encode('ascii', 'ignore')\
                       .decode('utf-8')

    # unificar separadores
    texto = texto.replace(',', ' ')
    texto = texto.replace('-', ' ')
    texto = texto.replace('/', ' ')
    texto = texto.replace('(', ' ')
    texto = texto.replace(')', ' ')
    texto = texto.replace('.', ' ')

    # eliminar espacios extra
    texto = re.sub(r'\s+', ' ', texto)

    return texto.strip()

In [ ]:
# Limpiar 'fertilizantes aplicados'
df_working_copy3['fertilizantes_aplicados_limpio'] = df_working_copy3['fertilizantes aplicados'].apply(limpiar_texto)

In [ ]:
# Palabras claces en toda la celda
# se busca en toda la celda la palabra clave por tipo de fretilización, pueden
# tener varios tipod de fertilización a la vez

keywords = {

    # FÓSFORO
    'aplica_p': [

        # fuentes directas
        'fosforo',
        'fosforica',
        'fosforita',
        'roca fosforica',
        'roca fosforia',
        'calfos',

        # fertilizantes fosfatados
        'dap',
        'super cafe',

        # formulaciones npk
        '15 15 15',
        '15 15 16',
        '15 15 17',

        '10 20 20',
        '10 20 30',
        '10 20 10',
        '10 30 10',
        '10 12 24',

        '12 24 24',
        '13 26 6',

        '16 16 16',

        '18 18 18',
        '18 18 19',

        '17 6 18 6',

        '15 15 15 10 10 10',

        # comerciales
        'triple 15',
        'triple 18',
        'tres triplex 18',

        'npk'
    ],


    # POTASIO

    'aplica_k': [

        # fuentes directas
        'kcl',
        'potasio',
        'n k',

        # formulaciones npk
        '15 15 15',
        '15 15 16',
        '15 15 17',

        '10 20 20',
        '10 20 30',
        '10 20 10',

        '16 16 16',

        '18 18 18',
        '18 18 19',

        '17 6 18 6',

        '15 15 15 10 10 10',

        '13 26 6',

        # comerciales
        'triple 15',
        'triple 18',
        'tres triplex 18',

        'npk',

        # orgánicos ricos en k
        'ceniza',
        'cenizas',
        'vinaza'
    ],

    # CAL Y ENMIENDAS

    'aplica_cal': [

        'cal',
        'cal dolomita',
        'caldolomita',
        'cal agricola',
        'cal viva',
        'dolomita'
    ],

    # ORGÁNICO

    'aplica_organico': [

        'organico',
        'organicos',

        'gallinaza',
        'porquinaza',
        'pollinaza',

        'boiga',
        'boniga',
        'boñiga',

        'compost',

        'vinaza',

        'biocane',

        'ceniza',
        'cenizas',

        'materia organica',

        'abono organico',

        'heces',
        'heces de vaca',

        'aboniza'
    ],
    # KCL ESPECÍFICO
    'aplica_kcl': [
        'kcl'
    ],

    # FERTILIZACIÓN GENERAL
    # NO SABES QUÉ APLICÓ,
    # PERO SÍ SABES QUE FERTILIZA

    'fert_generica': [

        'si',
        'si cada corte',

        'mezclas',

        'quimico',
        'quimico organico',

        'quimica',

        'fertimenores',

        'aboniza'
    ]
}

In [ ]:
# Crear las columnas binarias de cada tipo de fertilización
for columna in keywords.keys():
    df_working_copy3[columna] = 0
for columna, palabras in keywords.items():
    df_working_copy3[columna] = df_working_copy3['fertilizantes_aplicados_limpio'].apply(
        lambda texto: int(
            any(
                palabra in texto
                for palabra in palabras
            )
        )
    )

In [ ]:
# REVISAR RESULTADOS

print(df_working_copy3[[
    'fertilizantes aplicados',
    'fertilizantes_aplicados_limpio',
    'aplica_p',
    'aplica_k',
    'aplica_cal',
    'aplica_organico',
    'aplica_kcl',
    'fert_generica'

]].head(10))

  fertilizantes aplicados fertilizantes_aplicados_limpio  aplica_p  aplica_k  \
0           no fertilizan                  no fertilizan         0         0   
1           si cada corte                  si cada corte         0         0   
2           no fertilizan                  no fertilizan         0         0   
3           no fertilizan                  no fertilizan         0         0   
4           no fertilizan                  no fertilizan         0         0   
5           no fertilizan                  no fertilizan         0         0   
6           no fertilizan                  no fertilizan         0         0   
7           no fertilizan                  no fertilizan         0         0   
8                 mezclas                        mezclas         0         0   
9               no indica                      no indica         0         0   

   aplica_cal  aplica_organico  aplica_kcl  fert_generica  
0           0                0           0              0  

In [ ]:
# Cuantos se detectaron en cada tipo de fertilización
for columna in keywords.keys():

    print(f'\n{columna}')
    print(df_working_copy3[columna].value_counts())


aplica_p
aplica_p
0    2875
1     383
Name: count, dtype: int64

aplica_k
aplica_k
0    2802
1     456
Name: count, dtype: int64

aplica_cal
aplica_cal
0    3216
1      42
Name: count, dtype: int64

aplica_organico
aplica_organico
0    3041
1     217
Name: count, dtype: int64

aplica_kcl
aplica_kcl
0    3235
1      23
Name: count, dtype: int64

fert_generica
fert_generica
0    2791
1     467
Name: count, dtype: int64


In [ ]:
# ELIMINAR COLUMNAS ORIGINAL Y LIMPIA TEMPORAL

df_working_copy3 = df_working_copy3.drop(columns=['fertilizantes aplicados', 'fertilizantes_aplicados_limpio'])

print("\nDataFrame después de eliminar las columnas 'fertilizantes aplicados' y 'fertilizantes_aplicados_limpio':")
display(df_working_copy3.head())


DataFrame después de eliminar las columnas 'fertilizantes aplicados' y 'fertilizantes_aplicados_limpio':


,ph agua:suelo,materia organica,fósforo bray ii,capacidad de intercambio cationico,conductividad electrica,potasio intercambiable,departamento_encoded,municipio_encoded,tiempo_establecimiento_encoded,topografia_encoded,riego_encoded,drenaje_encoded,aplica_p,aplica_k,aplica_cal,aplica_organico,aplica_kcl,fert_generica
0,7.32,6.258,58.11,21.80,2.123,0.348,9,132,4,2,0.0,0,0,0,0,0,0,0
1,6.44,5.935,43.24,17.63,0.742,0.284,9,132,4,2,0.0,0,0,0,0,0,0,1
2,5.91,4.688,6.999,18.61,1.196,0.246,9,132,4,2,0.0,0,0,0,0,0,0,0
3,5.63,3.995,21.31,8.906,0.310,0.220,9,77,4,2,0.0,0,0,0,0,0,0,0
4,5.65,4.113,84.32,10.39,0.247,0.435,9,78,4,2,0.0,0,0,0,0,0,0,0


## Datos numéricos

In [ ]:
# Crear una nueva copia del DataFrame para esta etapa de procesamiento
df_working_copy4 = df_working_copy3.copy()

In [ ]:
df_working_copy4.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3258 entries, 0 to 3257
Data columns (total 18 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   ph agua:suelo                       3253 non-null   float64
 1   materia organica                    3253 non-null   float64
 2   fósforo bray ii                     3244 non-null   float64
 3   capacidad de intercambio cationico  3252 non-null   float64
 4   conductividad electrica             3248 non-null   float64
 5   potasio intercambiable              3253 non-null   float64
 6   departamento_encoded                3258 non-null   int64  
 7   municipio_encoded                   3258 non-null   int64  
 8   tiempo_establecimiento_encoded      3258 non-null   int64  
 9   topografia_encoded                  3258 non-null   int64  
 10  riego_encoded                       3255 non-null   float64
 11  drenaje_encoded                     3258 no

In [ ]:
columns_to_convert_to_category = [
    'departamento_encoded',
    'municipio_encoded',
    'tiempo_establecimiento_encoded',
    'topografia_encoded',
    'riego_encoded',
    'drenaje_encoded',
    'aplica_p',
    'aplica_k',
    'aplica_cal',
    'aplica_organico',
    'aplica_kcl',
    'fert_generica'
]

for col in columns_to_convert_to_category:
    df_working_copy4[col] = df_working_copy4[col].astype('category')

print("DataFrame info after converting specified columns to 'category' type:")
df_working_copy4.info()

DataFrame info after converting specified columns to 'category' type:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3258 entries, 0 to 3257
Data columns (total 18 columns):
 #   Column                              Non-Null Count  Dtype   
---  ------                              --------------  -----   
 0   ph agua:suelo                       3253 non-null   float64 
 1   materia organica                    3253 non-null   float64 
 2   fósforo bray ii                     3244 non-null   float64 
 3   capacidad de intercambio cationico  3252 non-null   float64 
 4   conductividad electrica             3248 non-null   float64 
 5   potasio intercambiable              3253 non-null   float64 
 6   departamento_encoded                3258 non-null   category
 7   municipio_encoded                   3258 non-null   category
 8   tiempo_establecimiento_encoded      3258 non-null   category
 9   topografia_encoded                  3258 non-null   category
 10  riego_encoded             

In [ ]:
import os

save_directory = '/content/drive/MyDrive/Semestre9/agriculturaPredictiva/Final_predictiva/'
file_name = 'df_processed.csv'
full_save_path = os.path.join(save_directory, file_name)

# Ensure the directory exists (though for MyDrive it usually does)
os.makedirs(save_directory, exist_ok=True)

# Save the DataFrame as a CSV file
df_working_copy4.to_csv(full_save_path, index=False)

print(f"DataFrame guardado exitosamente en: {full_save_path}")

DataFrame guardado exitosamente en: /content/drive/MyDrive/Semestre9/agriculturaPredictiva/Final_predictiva/df_processed.csv
